# LLM API Tutorial (OpenAI, Anthropic, Gemini)

This tutorial demonstrates the **common structure** for using modern LLM APIs.

We will progressively build:

1. Basic API call
2. Add chat history
3. Add native web search


## 0. Setup Environment

## 1. Install Dependencies

# 2. Environment Variables

Create a `.env` file:


```env
OPENAI_API_KEY=your_openai_key
ANTHROPIC_API_KEY=your_anthropic_key
GEMINI_API_KEY=your_google_key
```

## 3. Common API Pattern

Use this mental model for every provider:

```text
client
   handles connection

model
   chooses capability

system prompt
   defines behavior

tools
   gives abilities

config
   tunes generation

user question
   defines task
```


Recommended structure:

```python
# 1. Client
client = ProviderClient()

# 2. Model
MODEL = "..."

# 3. Default prompt
SYSTEM_PROMPT = "You are a helpful assistant."

# 4. Optional tools
TOOLS = [...]

# 5. Optional generation settings
CONFIG = {...}

# 6. Dynamic user question
USER_QUESTION = "..."

# 7. API request
response = ...
```

## Part 1 — OpenAI

Official docs: https://platform.openai.com/docs

### 1.1 Basic API Call

In [12]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# -------------------
# 1. Client
# -------------------
client = OpenAI()

# -------------------
# 2. Model
# -------------------
MODEL = "gpt-4o-mini"

# -------------------
# 3. System Prompt
# -------------------
SYSTEM_PROMPT = "You are a helpful assistant."

# -------------------
# 4. Tools
# -------------------
TOOLS = []

# -------------------
# 5. Config
# -------------------
CONFIG = {
    "temperature": 0.7
}

# -------------------
# 6. User Question
# -------------------
USER_QUESTION = "Explain what an LLM is."

# -------------------
# 7. Request
# -------------------
response = client.responses.create(
    model=MODEL,
    instructions=SYSTEM_PROMPT,
    tools=TOOLS,
    input=USER_QUESTION,
    **CONFIG
)

print(response.output_text)

An LLM, or Large Language Model, is a type of artificial intelligence designed to understand and generate human language. These models are based on deep learning techniques, particularly using neural networks, and are trained on vast amounts of text data from books, articles, websites, and other sources.

### Key Features of LLMs:

1. **Scale**: LLMs are characterized by their large number of parameters, often in the billions or even trillions, which allows them to capture complex patterns and nuances in language.

2. **Training**: They are trained using unsupervised or semi-supervised learning methods, where the model learns to predict the next word in a sentence given the previous words. This process helps the model gain a broad understanding of language structure and context.

3. **Contextual Understanding**: LLMs can understand context, which allows them to generate coherent and contextually relevant responses. They can maintain context over longer conversations, making them useful

#### `temperature`

| Value  | Behavior                |
| ------ | ----------------------- |
| `0.0`  | deterministic           |
| `0.2`  | focused                 |
| `0.7`  | balanced (good default) |
| `1.0`  | creative                |
| `>1.2` | more random             |

#### `max_output_token`

| Task            | Suggested   |
| --------------- | ----------- |
| Short answers   | `200`       |
| Explanation     | `500`       |
| Long tutorial   | `1000–2000` |
| Code generation | `1500+`     |


In [23]:
client = OpenAI()

MODEL = "gpt-4o-mini"
SYSTEM_PROMPT = "You are a helpful assistant."
TOOLS = [
    {
        "type": "web_search"
    }
]
CONFIG = {
    "temperature": 0.2,
    "max_output_tokens": 1000,
}

history = []

while True:
    USER_QUESTION = input("Enter your message ('exit' or 'quit'): ")

    if USER_QUESTION.lower() in ["exit", "quit"]:
        print("Exiting...")
        break

    print("User:", USER_QUESTION)
    
    history.append({
        "role": "user",
        "content": USER_QUESTION
    })

    response = client.responses.create(
        model=MODEL,
        instructions=SYSTEM_PROMPT,
        tools=TOOLS,
        input=history,
        **CONFIG
    )

    answer = response.output_text
    print("AI:", answer)

    history.append({
        "role": "assistant",
        "content": answer
    })

User: My name is Thomas, what is your name
AI: I'm called Assistant! How can I help you today, Thomas?
User: What is yoru model based on?
AI: I'm based on a model developed by OpenAI called GPT (Generative Pre-trained Transformer). It's designed to understand and generate human-like text based on the input it receives. If you have any specific questions about how it works or its capabilities, feel free to ask!
User: which version
AI: I'm based on the GPT-4 architecture. If you have any questions about its features or capabilities, let me know!
User: who win the supoer bowl 2026
AI: 
## NFL Schedule
- Seattle Seahawks (SEA) @ New England Patriots (NE) on Sunday, Feb 08, 2026 at 03:30 PM PST. Final score: SEA 29 - NE 13


The Seattle Seahawks secured their second Super Bowl title by defeating the New England Patriots 29-13 in Super Bowl LX on February 8, 2026, at Levi's Stadium in Santa Clara, California. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Super_Bowl_LX?utm_source=openai))

### ChatGPT API Changes for GPT-5

Here’s a **clean, accurate summary of GPT-4 vs GPT-5 API changes** (based on the latest Responses API design and GPT-5 docs).

---

# 🧠 GPT-4 vs GPT-5 API — Key Differences

## 1. Core design shift

### GPT-4 family (gpt-4o, gpt-4o-mini)

```text
Goal: generate text
Control: randomness
```

You control behavior with:

* `temperature`
* `top_p`
* `max_tokens`

---

### GPT-5 family (gpt-5, gpt-5-nano, etc.)

```text
Goal: reason + generate
Control: thinking effort + verbosity
```

You control behavior with:

* `reasoning.effort`
* `text.verbosity`
* `max_output_tokens`

📌 Source: GPT-5 API introduces reasoning + verbosity controls ([OpenAI][1])

---

# 2. Parameter changes (IMPORTANT)

## ❌ Removed / not used the same way

### GPT-5 does NOT rely on:

* `temperature` (not primary control anymore for GPT-5 reasoning models)
* `top_p` (rarely used in GPT-5 reasoning flow)

👉 GPT-5 shifts away from randomness tuning.

---

## ✅ New GPT-5 parameters

### 1. Reasoning control (NEW)

```python
"reasoning": {
    "effort": "minimal | low | medium | high"
}
```

Think:

```text
How hard should the model think?
```

📌 GPT-5 supports configurable reasoning effort levels ([OpenAI][1])

---

### 2. Output style control (NEW)

```python
"text": {
    "verbosity": "low | medium | high"
}
```

Think:

```text
How detailed should the answer be?
```

📌 GPT-5 adds verbosity control for response length and detail ([OpenAI][1])

---

### 3. Output length (same concept)

```python
"max_output_tokens": 500
```

Still controls hard token limit.

---

# 3. API structure change

## GPT-4 style (classic)

```python
response = client.responses.create(
    model="gpt-4o-mini",
    input="Hello",
    temperature=0.7,
    max_output_tokens=500
)
```

---

## GPT-5 style (modern Responses API)

```python
response = client.responses.create(
    model="gpt-5-nano",
    input="Hello",
    reasoning={
        "effort": "minimal"
    },
    text={
        "verbosity": "medium"
    },
    max_output_tokens=500
)
```

---

# 4. Mental model shift (VERY IMPORTANT)

## GPT-4 thinking

```text
“Should I be more random or deterministic?”
```

→ temperature-driven

---

## GPT-5 thinking

```text
“How much should I reason before answering?”
“How detailed should the output be?”
```

→ reasoning-driven

---

# 5. Model differences (API perspective)

## GPT-4 models

* gpt-4o
* gpt-4o-mini

Characteristics:

* fast
* flexible
* simpler control
* strong chat behavior

Best for:

* chatbots
* summarization
* general assistants

---

## GPT-5 models

* gpt-5
* gpt-5-mini
* gpt-5-nano

Characteristics:

* explicit reasoning control
* better multi-step reasoning
* structured output alignment improved
* more predictable “thinking level”

📌 GPT-5 is optimized for coding + agentic + reasoning tasks ([OpenAI][1])

---

# 6. Tool calling differences

## GPT-4

```python
tools=[{"type": "function"}]
```

---

## GPT-5

Same concept but:

* more reliable tool selection
* supports custom tools + built-in tools better
* reasoning happens before tool calls (more structured)

---

# 7. Key practical differences (summary table)

| Feature           | GPT-4 (4o / 4o-mini) | GPT-5 (gpt-5 / nano)        |
| ----------------- | -------------------- | --------------------------- |
| Primary control   | temperature          | reasoning.effort            |
| Output control    | implicit             | verbosity                   |
| API style         | simpler              | structured config objects   |
| Thinking behavior | hidden               | explicit                    |
| Best use          | chat + general tasks | reasoning + agents + coding |
| Predictability    | medium               | high                        |

---

# 8. Final “proper syntax” templates

## GPT-4o / GPT-4o-mini (classic)

```python
CONFIG = {
    "temperature": 0.7,
    "max_output_tokens": 500
}
```

---

## GPT-5 / GPT-5-nano (modern)

```python
CONFIG = {
    "reasoning": {
        "effort": "minimal"
    },
    "text": {
        "verbosity": "medium"
    },
    "max_output_tokens": 500
}
```

---

# 9. Key takeaway (important)

### GPT-4 mindset:

> “Control randomness”

### GPT-5 mindset:

> “Control thinking depth + output style”



In [35]:
client = OpenAI()

MODEL = "gpt-5-nano"
SYSTEM_PROMPT = "You are a helpful assistant."
TOOLS = [
    {
        "type": "web_search"
    }
]
CONFIG = {
    "reasoning": {
        "effort": "low"
    },
    "text": {
        "verbosity": "medium"
    },
    "max_output_tokens": 500
}

history = []

while True:
    USER_QUESTION = input("Enter your message ('exit' or 'quit'): ")

    if USER_QUESTION.lower() in ["exit", "quit"]:
        print("Exiting...")
        break

    print("User:", USER_QUESTION)
    
    history.append({
        "role": "user",
        "content": USER_QUESTION
    })

    response = client.responses.create(
        model=MODEL,
        instructions=SYSTEM_PROMPT,
        tools=TOOLS,
        input=history,
        **CONFIG
    )

    answer = response.output_text
    print("AI:", answer)

    history.append({
        "role": "assistant",
        "content": answer
    })

User: hi
AI: Hi there! How can I help today? If you’re not sure where to start, tell me a bit about what you’re working on or what you’re curious about, and I’ll jump in.
User: my name is thomas. what is the latest news in april 2026
AI: Hi Thomas — happy to help. Since you asked for the “latest news in April 2026,” here’s a quick snapshot of notable headlines and events from that month. If you want deeper coverage on a specific topic (space, politics, economy, etc.), tell me and I’ll tailor it.

- Space and aerospace
  - Artemis II mission: early April 2026 was highlighted as NASA’s crewed moon mission with media attention on an April launch window and mission readiness. Multiple outlets framed it as a historic step in crewed lunar exploration. ([livescience.com](https://www.livescience.com/space/space-exploration/something-really-big-is-going-to-happen-nasas-historic-artemis-ii-mission-approved-for-april-1-launch?utm_source=openai))
  - Lyrid meteor shower: Space.com covered striking


# Part 2 — Anthropic Claude

Official docs: https://docs.anthropic.com

## 2.1 Basic API Call


In [2]:
import anthropic
from dotenv import load_dotenv

load_dotenv()

# -------------------
# 1. Client
# -------------------
client = anthropic.Anthropic()

# -------------------
# 2. Model
# -------------------
MODEL = "claude-sonnet-4-6"

# -------------------
# 3. System Prompt
# -------------------
SYSTEM_PROMPT = "You are a helpful assistant."

# -------------------
# 4. Tools
# -------------------
TOOLS = []

# -------------------
# 5. Config
# -------------------
CONFIG = {
    "max_tokens": 500,
    "temperature": 0.3
}

# -------------------
# 6. User Question
# -------------------
USER_QUESTION = "Explain transformers simply."

# -------------------
# 7. Request
# -------------------
response = client.messages.create(
    model=MODEL,
    system=SYSTEM_PROMPT,
    messages=[
        {
            "role": "user",
            "content": USER_QUESTION
        }
    ],
    **CONFIG
)

print(response.content[0].text)


# Transformers (AI) Simply Explained

## The Core Problem
When processing language, words relate to each other across long distances. Traditional AI struggled with this - it processed words one at a time and "forgot" earlier context.

---

## The Key Idea: Attention

Transformers ask: **"Which words should I pay attention to when understanding this word?"**

**Example:**
> *"The animal didn't cross the street because **it** was too tired"*

What does "it" refer to? The **animal**, not the street. Transformers figure this out by letting every word look at every other word simultaneously.

---

## How It Works (Simply)

1. **Input** - Break text into tokens (words/pieces)
2. **Embed** - Convert tokens into numbers
3. **Attention** - Each token asks *"how relevant is every other token to me?"*
4. **Score & Weight** - Relevant tokens get more influence
5. **Output** - Produce a result based on the full context

---

## Why It Was Revolutionary

| Old Approach | Transformer |
|---|---|
| Se

# Part 3 — Google Gemini

Official docs: https://ai.google.dev/gemini-api/docs


## 3.1 Basic API Call

In [31]:
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()

# -------------------
# 1. Client
# -------------------
client = genai.Client()

# -------------------
# 2. Model
# -------------------
MODEL = "gemini-2.5-flash-lite"

# -------------------
# 3. System Prompt
# -------------------
SYSTEM_PROMPT = "You are a helpful assistant."

# -------------------
# 4. Config
# -------------------
CONFIG = types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT,
    temperature=0.2
)

# -------------------
# 5. User Question
# -------------------
USER_QUESTION = "Explain embeddings simply."

# -------------------
# 6. Request
# -------------------
response = client.models.generate_content(
    model=MODEL,
    contents=USER_QUESTION,
    config=CONFIG
)

print(response.text)


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Imagine you have a bunch of words, like "apple," "banana," "car," and "truck."

**Embeddings are like giving each of these words a unique "address" in a special kind of space.**

Think of this space as a multi-dimensional map. Instead of just north/south or east/west, this map has many directions.

Here's the key idea:

*   **Similar words get similar addresses.** Words that mean similar things will be located close to each other on this map. So, "apple" and "banana" might be neighbors, while "car" and "truck" are also close.
*   **Different words get different addresses.** Words that have very different meanings will be far apart on the map. "Apple" and "car" would be quite distant.

**Why is this useful?**

Computers don't understand words like we do. They understand numbers. Embeddings translate words into lists of numbers (their "address"). This allows computers to:

*   **Understand relationships between words:** By looking at the "addresses," a computer can figure out that "king"

## Side-by-Side Comparison

| Concept | OpenAI | Anthropic | Gemini |
|---|---|---|---|
| System Prompt | `instructions=` | `system=` | `system_instruction=` |
| Model | `model=` | `model=` | `model=` |
| Tools | `tools=` | `tools=` | `tools=` |
| Config | `dict` | `dict` | `GenerateContentConfig()` |
| User Input | `input=` | `messages=` | `contents=` |
| Chat History | Manual | Manual | Native Chat |
| Web Search | `web_search` | `web_search` | `GoogleSearch()` |




# Key Takeaways

Always structure your LLM code like this:

```python
client
MODEL
SYSTEM_PROMPT
TOOLS
CONFIG
USER_QUESTION
response
```

This keeps your code:

- readable
- reusable
- provider-independent
- easy to upgrade

The API syntax changes, but the architecture stays the same.